##### SparkSession 

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PropTech360-Test")
    .master("local[*]")
    .getOrCreate()
)

print(spark.version)

spark.stop()

c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [4]:
# Environment check: record the versions in your README.
import sys
import importlib.metadata as metadata
print('Python:', sys.version.split()[0])
for package in ['pandas', 'numpy', 'pyspark', 'pyarrow', 'sqlalchemy', 'psycopg', 'boto3']:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: NOT INSTALLED (install before its assessed section)')

Python: 3.12.10
pandas: 3.0.6
numpy: 2.5.3
pyspark: 4.2.0
pyarrow: 25.0.1
sqlalchemy: 2.1.3
psycopg: 3.3.6
boto3: 1.43.108


In [5]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import pandas as pd

In [6]:
EXPECTED_COLUMNS = {
    "properties.csv": {
        "property_id",
        "property_name",
        "city"
    },

    "units.csv": {
        "unit_id",
        "property_id",
        "floor_area_sqm",
        "active"
    },

    "leases.csv": {
        "lease_id",
        "unit_id",
        "start_date",
        "end_date",
        "monthly_rent_usd"
    },

    "work_orders.csv": {
        "work_order_id",
        "unit_id",
        "opened_date",
        "closed_date",
        "priority",
        "response_hours",
        "cost_usd",
        "status"
    },

    "meter_readings.csv": {
        "reading_id",
        "unit_id",
        "reading_month",
        "energy_kwh"
    }
}

#### Verify the actual column names because your file may use name rather than proprty_name

In [7]:
def ingest_source(path, expected_columns):
    if not path.exists():
        raise FileNotFoundError(...)

    df = pd.read_csv(path)

    if df.empty:
        raise ValueError(...)

    missing_columns = expected_columns - set(df.columns)

    if missing_columns:
        raise ValueError(...)

    # hash file
    # create timestamp
    # return dataframe + manifest information

In [8]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import pandas as pd


def ingest_source(path: Path, expected_columns: set):
    """
    Validate a CSV source file, load it with stable string dtypes,
    and return both the DataFrame and ingestion metadata.
    """

    # 1. Validate that the file exists
    if not path.exists():
        raise FileNotFoundError(
            f"Source file does not exist: {path}"
        )

    # 2. Validate that the path is actually a file
    if not path.is_file():
        raise ValueError(
            f"Expected a file but received: {path}"
        )

    # 3. Validate that the file is not empty
    if path.stat().st_size == 0:
        raise ValueError(
            f"Source file is empty: {path}"
        )

    # 4. Load using stable dtypes
    # Keep source values as strings during ingestion.
    # Numeric/date conversion should happen later during validation/cleaning.
    try:
        df = pd.read_csv(
            path,
            dtype="string",
            keep_default_na=True
        )
    except Exception as exc:
        raise ValueError(
            f"Could not read CSV file {path.name}: {exc}"
        ) from exc

    # 5. Validate that records exist
    if df.empty:
        raise ValueError(
            f"Source file contains no data rows: {path}"
        )

    # 6. Validate required columns
    actual_columns = set(df.columns)

    missing_columns = expected_columns - actual_columns

    if missing_columns:
        raise ValueError(
            f"{path.name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    # 7. Compute SHA-256 hash of the original file
    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    file_hash = sha256.hexdigest()

    # 8. Create UTC ingestion timestamp
    ingest_timestamp = datetime.now(timezone.utc).isoformat()

    # 9. Create manifest record
    manifest_entry = {
        "filename": path.name,
        "file_path": str(path),
        "row_count": len(df),
        "column_count": len(df.columns),
        "ingest_timestamp_utc": ingest_timestamp,
        "sha256": file_hash
    }

    return df, manifest_entry

In [9]:
datasets = {}
ingestion_manifest = []

In [10]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import pandas as pd


def ingest_source(path, expected_columns):
    """
    Load a CSV source file, validate required columns,
    and create ingestion metadata.
    """

    path = Path(path)

    # 1. Check file exists
    if not path.exists():
        raise FileNotFoundError(
            f"Source file does not exist: {path.resolve()}"
        )

    # 2. Read CSV
    df = pd.read_csv(path)

    # 3. Validate expected columns
    missing_columns = set(expected_columns) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{path.name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    # 4. Calculate SHA-256 hash
    sha256 = hashlib.sha256(path.read_bytes()).hexdigest()

    # 5. Create ingestion manifest
    manifest = {
        "file_name": path.name,
        "file_path": str(path.resolve()),
        "row_count": len(df),
        "column_count": len(df.columns),
        "sha256": sha256,
        "ingested_at_utc": datetime.now(timezone.utc).isoformat()
    }

    return df, manifest

#### Test one of the actual file

In [11]:
RAW_DIR = Path(r"..data\raw\properties.csv")

In [12]:
RAW_DIR = Path("..") / "data" / "raw"

In [13]:
df, manifest = ingest_source(
    RAW_DIR / "properties.csv",
    {"property_id", "property_name", "city"}
)

In [14]:
print(df.head())

  property_id        property_name           city
0        P001  MetroSpace Tower 01          Lagos
1        P002  MetroSpace Tower 02          Abuja
2        P003  MetroSpace Tower 03         Ibadan
3        P004  MetroSpace Tower 04  Port Harcourt
4        P005  MetroSpace Tower 05          Lagos


In [15]:
print(manifest)

{'file_name': 'properties.csv', 'file_path': 'C:\\Documents\\Desktop\\PropTech360_Capstone_data_pipeline\\data\\raw\\properties.csv', 'row_count': 12, 'column_count': 3, 'sha256': 'c9865bcf2b142026c5dd05547f9c14d3e11f5a27a3e22bca52f5efe0c795ef19', 'ingested_at_utc': '2026-10-07T18:00:23.895660+00:00'}


In [16]:
from pathlib import Path

for file in Path(".").rglob("properties.csv"):
    print(file.resolve())

C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\notebook\data\staging\properties.csv


In [17]:
RAW_DIR = Path("data/raw")

In [18]:
{
    "filename": "properties.csv",
    "file_path": "data/raw/properties.csv",
    "row_count": 12,
    "column_count": 3,
    "ingest_timestamp_utc": "2026-10-03T...",
    "sha256": "..."
}

{'filename': 'properties.csv',
 'file_path': 'data/raw/properties.csv',
 'row_count': 12,
 'column_count': 3,
 'ingest_timestamp_utc': '2026-10-03T...',
 'sha256': '...'}

In [19]:
from pathlib import Path

RAW_DIR = Path("..") / "data" / "raw"

In [20]:
datasets = {}
ingestion_manifest = []

for filename, expected_columns in EXPECTED_COLUMNS.items():

    file_path = RAW_DIR / filename

    df, manifest_entry = ingest_source(
        file_path,
        expected_columns
    )

    datasets[filename] = df
    ingestion_manifest.append(manifest_entry)

In [21]:
print("Current working directory:")
print(Path.cwd())

print("\nRaw directory:")
print(RAW_DIR.resolve())

print("\nProperties path:")
print((RAW_DIR / "properties.csv").resolve())

print("\nDoes it exist?")
print((RAW_DIR / "properties.csv").exists())

Current working directory:
c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\notebook

Raw directory:
C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\raw

Properties path:
C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\raw\properties.csv

Does it exist?
True


In [22]:
for filename, df in datasets.items():
    print(filename, df.shape)

properties.csv (12, 3)
units.csv (241, 4)
leases.csv (198, 5)
work_orders.csv (1501, 8)
meter_readings.csv (1440, 4)


In [23]:
import pandas as pd

test_df = pd.read_csv(RAW_DIR / "properties.csv")
print(test_df.columns.tolist())

['property_id', 'property_name', 'city']


In [24]:
ingestion_manifest

[{'file_name': 'properties.csv',
  'file_path': 'C:\\Documents\\Desktop\\PropTech360_Capstone_data_pipeline\\data\\raw\\properties.csv',
  'row_count': 12,
  'column_count': 3,
  'sha256': 'c9865bcf2b142026c5dd05547f9c14d3e11f5a27a3e22bca52f5efe0c795ef19',
  'ingested_at_utc': '2026-10-07T18:00:23.975501+00:00'},
 {'file_name': 'units.csv',
  'file_path': 'C:\\Documents\\Desktop\\PropTech360_Capstone_data_pipeline\\data\\raw\\units.csv',
  'row_count': 241,
  'column_count': 4,
  'sha256': 'f9bcef3d08d4662c270bfff385299306995ed6ba3f0c54e6b3754274a53c2748',
  'ingested_at_utc': '2026-10-07T18:00:23.986503+00:00'},
 {'file_name': 'leases.csv',
  'file_path': 'C:\\Documents\\Desktop\\PropTech360_Capstone_data_pipeline\\data\\raw\\leases.csv',
  'row_count': 198,
  'column_count': 5,
  'sha256': 'b6a8e10a11dfbca4358d5a9d60ad4709bc6511417137aeb58e389b21e31ebe0a',
  'ingested_at_utc': '2026-10-07T18:00:23.999501+00:00'},
 {'file_name': 'work_orders.csv',
  'file_path': 'C:\\Documents\\Deskto

#### Save the Manifest

In [25]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

manifest_path = CURATED_DIR / "ingestion_manifest.json"

with manifest_path.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        ingestion_manifest,
        file,
        indent=4,
        default=str
    )

print(f"Manifest saved to: {manifest_path.resolve()}")

Manifest saved to: C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\curated\ingestion_manifest.json


In [26]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CURATED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)
print("Curated directory:", CURATED_DIR)

Project root: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline
Raw directory: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\raw
Curated directory: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\curated


In [27]:
datasets.keys()

dict_keys(['properties.csv', 'units.csv', 'leases.csv', 'work_orders.csv', 'meter_readings.csv'])

In [28]:
datasets = {
    "properties": datasets["properties.csv"],
    "units": datasets["units.csv"],
    "leases": datasets["leases.csv"],
    "work_orders": datasets["work_orders.csv"],
    "meter_readings": datasets["meter_readings.csv"]
}

In [29]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

In [30]:
for name, df in datasets.items():
    print(name, df.shape)

properties (12, 3)
units (241, 4)
leases (198, 5)
work_orders (1501, 8)
meter_readings (1440, 4)


#### Creating identifier

In [31]:
IDENTIFIERS = {
    "properties.csv": "property_id",
    "units.csv": "unit_id",
    "leases.csv": "lease_id",
    "work_orders.csv": "work_order_id",
    "meter_readings.csv": "meter_reading_id"
}

#### Profile_table function

In [32]:
import pandas as pd


def profile_table(df: pd.DataFrame, identifier: str) -> dict:
    """
    Generate basic quality statistics for a dataframe.

    Returns:
        - row count
        - column count
        - null values by column
        - duplicate identifier count
        - pandas data types
    """

    profile = {
        "row_count": len(df),

        "column_count": len(df.columns),

        "nulls_by_column": df.isna().sum().to_dict(),

        "duplicate_identifier_count": (
            df.duplicated(subset=[identifier]).sum()
            if identifier in df.columns
            else None
        ),

        "data_types": {
            column: str(dtype)
            for column, dtype in df.dtypes.items()
        }
    }

    return profile

#### Test profiling function

In [33]:
properties_profile = profile_table(
    datasets["properties"],
    "property_id"
)

In [34]:
properties_profile

{'row_count': 12,
 'column_count': 3,
 'nulls_by_column': {'property_id': 0, 'property_name': 0, 'city': 0},
 'duplicate_identifier_count': np.int64(0),
 'data_types': {'property_id': 'str', 'property_name': 'str', 'city': 'str'}}

#### Profile all the five datasets

In [35]:
identifier = "properties_id"

properties_profile = profile_table(
    datasets["properties"],
    identifier
)

In [36]:
identifier = "leases_id"

leases_profile = profile_table(
    datasets["leases"],
    identifier
)

In [37]:
identifier = "unit_id"

units_profile = profile_table(
    datasets["units"],
    identifier
)

In [38]:
identifier = "meter_reading_id"

meter_readings_profile = profile_table(
    datasets["meter_readings"],
    identifier
)

In [39]:
identifier = "work_order_id"

work_orders_profile = profile_table(
    datasets["work_orders"],
    identifier
)

In [40]:
properties_profile = profile_table(
    datasets["properties"],
    "property_id"
)

units_profile = profile_table(
    datasets["units"],
    "unit_id"
)

leases_profile = profile_table(
    datasets["leases"],
    "lease_id"
)

work_orders_profile = profile_table(
    datasets["work_orders"],
    "work_order_id"
)

meter_readings_profile = profile_table(
    datasets["meter_readings"],
    "reading_id"
)

In [45]:
IDENTIFIERS = {
    "properties": "property_id",
    "units": "unit_id",
    "leases": "lease_id",
    "work_orders": "work_order_id",
    "meter_readings": "reading_id",
}

quality_profiles = {}

for dataset_name, df in datasets.items():
    identifier = IDENTIFIERS[dataset_name]

    quality_profiles[dataset_name] = profile_table(
        df,
        identifier
    )

In [46]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("Leases")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

Leases
Rows: 12
Duplicate IDs: 0
Nulls:
{'property_id': 0, 'property_name': 0, 'city': 0}
Leases
Rows: 241
Duplicate IDs: 1
Nulls:
{'unit_id': 0, 'property_id': 0, 'floor_area_sqm': 0, 'active': 0}
Leases
Rows: 198
Duplicate IDs: 0
Nulls:
{'lease_id': 0, 'unit_id': 0, 'start_date': 0, 'end_date': 139, 'monthly_rent_usd': 0}
Leases
Rows: 1501
Duplicate IDs: 1
Nulls:
{'work_order_id': 0, 'unit_id': 0, 'opened_date': 0, 'closed_date': 295, 'priority': 0, 'response_hours': 295, 'cost_usd': 0, 'status': 0}
Leases
Rows: 1440
Duplicate IDs: 0
Nulls:
{'reading_id': 0, 'unit_id': 0, 'reading_month': 0, 'energy_kwh': 0}


In [47]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("meter_readings")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

meter_readings
Rows: 12
Duplicate IDs: 0
Nulls:
{'property_id': 0, 'property_name': 0, 'city': 0}
meter_readings
Rows: 241
Duplicate IDs: 1
Nulls:
{'unit_id': 0, 'property_id': 0, 'floor_area_sqm': 0, 'active': 0}
meter_readings
Rows: 198
Duplicate IDs: 0
Nulls:
{'lease_id': 0, 'unit_id': 0, 'start_date': 0, 'end_date': 139, 'monthly_rent_usd': 0}
meter_readings
Rows: 1501
Duplicate IDs: 1
Nulls:
{'work_order_id': 0, 'unit_id': 0, 'opened_date': 0, 'closed_date': 295, 'priority': 0, 'response_hours': 295, 'cost_usd': 0, 'status': 0}
meter_readings
Rows: 1440
Duplicate IDs: 0
Nulls:
{'reading_id': 0, 'unit_id': 0, 'reading_month': 0, 'energy_kwh': 0}


In [48]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("properties")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

properties
Rows: 12
Duplicate IDs: 0
Nulls:
{'property_id': 0, 'property_name': 0, 'city': 0}
properties
Rows: 241
Duplicate IDs: 1
Nulls:
{'unit_id': 0, 'property_id': 0, 'floor_area_sqm': 0, 'active': 0}
properties
Rows: 198
Duplicate IDs: 0
Nulls:
{'lease_id': 0, 'unit_id': 0, 'start_date': 0, 'end_date': 139, 'monthly_rent_usd': 0}
properties
Rows: 1501
Duplicate IDs: 1
Nulls:
{'work_order_id': 0, 'unit_id': 0, 'opened_date': 0, 'closed_date': 295, 'priority': 0, 'response_hours': 295, 'cost_usd': 0, 'status': 0}
properties
Rows: 1440
Duplicate IDs: 0
Nulls:
{'reading_id': 0, 'unit_id': 0, 'reading_month': 0, 'energy_kwh': 0}


In [49]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("units")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

units
Rows: 12
Duplicate IDs: 0
Nulls:
{'property_id': 0, 'property_name': 0, 'city': 0}
units
Rows: 241
Duplicate IDs: 1
Nulls:
{'unit_id': 0, 'property_id': 0, 'floor_area_sqm': 0, 'active': 0}
units
Rows: 198
Duplicate IDs: 0
Nulls:
{'lease_id': 0, 'unit_id': 0, 'start_date': 0, 'end_date': 139, 'monthly_rent_usd': 0}
units
Rows: 1501
Duplicate IDs: 1
Nulls:
{'work_order_id': 0, 'unit_id': 0, 'opened_date': 0, 'closed_date': 295, 'priority': 0, 'response_hours': 295, 'cost_usd': 0, 'status': 0}
units
Rows: 1440
Duplicate IDs: 0
Nulls:
{'reading_id': 0, 'unit_id': 0, 'reading_month': 0, 'energy_kwh': 0}


In [50]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("work_orders")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

work_orders
Rows: 12
Duplicate IDs: 0
Nulls:
{'property_id': 0, 'property_name': 0, 'city': 0}
work_orders
Rows: 241
Duplicate IDs: 1
Nulls:
{'unit_id': 0, 'property_id': 0, 'floor_area_sqm': 0, 'active': 0}
work_orders
Rows: 198
Duplicate IDs: 0
Nulls:
{'lease_id': 0, 'unit_id': 0, 'start_date': 0, 'end_date': 139, 'monthly_rent_usd': 0}
work_orders
Rows: 1501
Duplicate IDs: 1
Nulls:
{'work_order_id': 0, 'unit_id': 0, 'opened_date': 0, 'closed_date': 295, 'priority': 0, 'response_hours': 295, 'cost_usd': 0, 'status': 0}
work_orders
Rows: 1440
Duplicate IDs: 0
Nulls:
{'reading_id': 0, 'unit_id': 0, 'reading_month': 0, 'energy_kwh': 0}


#### Helper for Quarantine reasons

In [51]:
def add_rejection_reason(
    df: pd.DataFrame,
    mask: pd.Series,
    reason: str
):
    """
    Add a rejection reason to rows matching mask.
    """

    df.loc[mask, "rejection_reason"] = (
        df.loc[mask, "rejection_reason"]
        .fillna("")
        .apply(
            lambda existing:
            f"{existing}; {reason}".strip("; ")
            if existing
            else reason
        )
    )

#### Validate and Quarantine

In [52]:
def validate_and_quarantine(
    datasets: dict,
    run_id: str
):

    clean_datasets = {}

    quarantine_datasets = {}

    quality_summary = {}

    return (
        clean_datasets,
        quarantine_datasets,
        quality_summary
    )

#### Validate properties data

In [53]:
properties = datasets["properties"].copy()

properties["rejection_reason"] = pd.NA

##### checking for duplicate (properties)

In [54]:
duplicate_property = properties.duplicated(
    subset=["property_id"],
    keep=False
)

add_rejection_reason(
    properties,
    duplicate_property,
    "duplicate_property_id"
)

#### checking for missing ID

In [55]:
missing_property_id = properties["property_id"].isna()

add_rejection_reason(
    properties,
    missing_property_id,
    "missing_property_id"
)

#### Validate units data

In [56]:
units = datasets["units"].copy()

units["rejection_reason"] = pd.NA

#### checking for duplicate (units)

In [57]:
duplicate_unit = units.duplicated(
    subset=["unit_id"],
    keep=False
)

add_rejection_reason(
    units,
    duplicate_unit,
    "duplicate_unit_id"
)

#### check unit (property foreign key)

In [58]:
valid_property_ids = set(
    properties.loc[
        properties["rejection_reason"].isna(),
        "property_id"
    ]
)

#### Invalid property references

In [59]:
unknown_property = (
    ~units["property_id"].isin(valid_property_ids)
)

add_rejection_reason(
    units,
    unknown_property,
    "unknown_property_id"
)

#### validate numeric floor area

In [60]:
units["floor_area_sqm"] = pd.to_numeric(
    units["floor_area_sqm"],
    errors="coerce"
)

In [61]:
invalid_floor_area = (
    units["floor_area_sqm"].isna()
    |
    (units["floor_area_sqm"] <= 0)
)

add_rejection_reason(
    units,
    invalid_floor_area,
    "invalid_floor_area"
)

#### Validate leases

In [62]:
leases = datasets["leases"].copy()

leases["rejection_reason"] = pd.NA

#### convert to dates & rent

In [63]:
leases["start_date"] = pd.to_datetime(
    leases["start_date"],
    errors="coerce"
)

leases["end_date"] = pd.to_datetime(
    leases["end_date"],
    errors="coerce"
)

In [64]:
leases["monthly_rent_usd"] = pd.to_numeric(
    leases["monthly_rent_usd"],
    errors="coerce"
)

#### Detect bad lease dates

In [65]:
bad_start_date = leases["start_date"].isna()

add_rejection_reason(
    leases,
    bad_start_date,
    "invalid_start_date"
)

#### For end dates, you might allow missing end dates for active leases. But if an end date exists and it is before the start

In [66]:
bad_date_order = (
    leases["end_date"].notna()
    &
    leases["start_date"].notna()
    &
    (leases["end_date"] < leases["start_date"])
)

add_rejection_reason(
    leases,
    bad_date_order,
    "end_date_before_start_date"
)

#### Unit foreign key validation (Lease)

In [67]:
valid_unit_ids = set(
    units.loc[
        units["rejection_reason"].isna(),
        "unit_id"
    ]
)

In [68]:
unknown_lease_unit = (
    ~leases["unit_id"].isin(valid_unit_ids)
)

add_rejection_reason(
    leases,
    unknown_lease_unit,
    "unknown_unit_id"
)

#### Validate rent

In [69]:
invalid_rent = (
    leases["monthly_rent_usd"].isna()
    |
    (leases["monthly_rent_usd"] < 0)
)

add_rejection_reason(
    leases,
    invalid_rent,
    "invalid_monthly_rent"
)

#### check duplicate lease IDs

In [70]:
duplicate_lease = leases.duplicated(
    subset=["lease_id"],
    keep=False
)

add_rejection_reason(
    leases,
    duplicate_lease,
    "duplicate_lease_id"
)

#### Validate work orders

In [71]:
work_orders = datasets["work_orders"].copy()

work_orders["rejection_reason"] = pd.NA

In [72]:
duplicate_work_order = work_orders.duplicated(
    subset=["work_order_id"],
    keep=False
)

add_rejection_reason(
    work_orders,
    duplicate_work_order,
    "duplicate_work_order_id"
)

#### Validate work-order unit

In [73]:
unknown_work_order_unit = (
    ~work_orders["unit_id"].isin(valid_unit_ids)
)

add_rejection_reason(
    work_orders,
    unknown_work_order_unit,
    "unknown_unit_id"
)

##### validate response hours

In [74]:
work_orders["response_hours"] = pd.to_numeric(
    work_orders["response_hours"],
    errors="coerce"
)

In [75]:
missing_response = (
    work_orders["response_hours"].isna()
)

add_rejection_reason(
    work_orders,
    missing_response,
    "missing_response_hours"
)

#### checking for negative hours

In [76]:
negative_response = (
    work_orders["response_hours"] < 0
)

add_rejection_reason(
    work_orders,
    negative_response,
    "negative_response_hours"
)

#### Validate work-order date

In [77]:
date_column = "created_date"

if date_column in work_orders.columns:
    work_orders[date_column] = pd.to_datetime(
        work_orders[date_column],
        errors="coerce"
    )
else:
    print(f"Column '{date_column}' not found.")
    print("Available columns:", work_orders.columns.tolist())

Column 'created_date' not found.
Available columns: ['work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status', 'rejection_reason']


In [78]:
print(datasets["work_orders"].columns.tolist())

['work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status']


#### Validate meter-readings

In [79]:
meter_readings = datasets[
    "meter_readings"
].copy()

meter_readings["rejection_reason"] = pd.NA

In [80]:
meter_readings["energy_kwh"] = pd.to_numeric(
    meter_readings["energy_kwh"],
    errors="coerce"
)

#### Reject invalid numeric values

In [81]:
invalid_kwh = meter_readings["energy_kwh"].isna()

add_rejection_reason(
    meter_readings,
    invalid_kwh,
    "invalid_kwh"
)

#### Detecting negative energy_kwh

In [82]:
negative_kwh = meter_readings["energy_kwh"] < 0

add_rejection_reason(
    meter_readings,
    negative_kwh,
    "negative_kwh"
)


####  checking unit foreign key (metering_reading)

In [83]:
unknown_meter_unit = (
    ~meter_readings["unit_id"].isin(valid_unit_ids)
)

add_rejection_reason(
    meter_readings,
    unknown_meter_unit,
    "unknown_unit_id"
)

#### validate meter_month

In [84]:
print(datasets["meter_readings"].columns.tolist())

['reading_id', 'unit_id', 'reading_month', 'energy_kwh']


In [85]:
meter_readings["reading_month"] = pd.to_datetime(
    meter_readings["reading_month"],
    errors="coerce"
)

In [86]:
bad_meter_date = (
    meter_readings["reading_month"].isna()
)

add_rejection_reason(
    meter_readings,
    bad_meter_date,
    "invalid_reading_date"
)

##### cleaning vs quarantine records

In [87]:
def split_clean_and_quarantine(df):

    clean = df[
        df["rejection_reason"].isna()
    ].copy()

    quarantine = df[
        df["rejection_reason"].notna()
    ].copy()

    clean = clean.drop(
        columns=["rejection_reason"]
    )

    return clean, quarantine

#### Validate and quarantine () function

In [88]:
import pandas as pd


def add_rejection_reason(
    df: pd.DataFrame,
    mask: pd.Series,
    reason: str
):

    current = df.loc[
        mask,
        "rejection_reason"
    ].fillna("")

    df.loc[
        mask,
        "rejection_reason"
    ] = current.apply(
        lambda x:
        f"{x}; {reason}".strip("; ")
        if x
        else reason
    )


def split_clean_and_quarantine(df):

    clean = df[
        df["rejection_reason"].isna()
    ].copy()

    quarantine = df[
        df["rejection_reason"].notna()
    ].copy()

    clean.drop(
        columns=["rejection_reason"],
        inplace=True
    )

    return clean, quarantine


def validate_and_quarantine(
    datasets: dict,
    run_id: str
):

    clean_datasets = {}
    quarantine_datasets = {}
    quality_summary = {}


    # ==================================
    # 1. PROPERTIES
    # ==================================

    properties = datasets[
        "properties"
    ].copy()

    properties["rejection_reason"] = pd.NA


    duplicate_property = properties.duplicated(
        subset=["property_id"],
        keep=False
    )

    add_rejection_reason(
        properties,
        duplicate_property,
        "duplicate_property_id"
    )


    missing_property = (
        properties["property_id"].isna()
    )

    add_rejection_reason(
        properties,
        missing_property,
        "missing_property_id"
    )


    properties_clean, properties_bad = (
        split_clean_and_quarantine(properties)
    )


    valid_property_ids = set(
        properties_clean["property_id"]
    )


    # ==================================
    # 2. UNITS
    # =================================]

    units = datasets[
        "units"
    ].copy()

    units["rejection_reason"] = pd.NA


    duplicate_unit = units.duplicated(
        subset=["unit_id"],
        keep=False
    )

    add_rejection_reason(
        units,
        duplicate_unit,
        "duplicate_unit_id"
    )


    unknown_property = (
        ~units["property_id"]
        .isin(valid_property_ids)
    )

    add_rejection_reason(
        units,
        unknown_property,
        "unknown_property_id"
    )


    units["floor_area_sqm"] = pd.to_numeric(
        units["floor_area_sqm"],
        errors="coerce"
    )


    invalid_floor_area = (
        units["floor_area_sqm"].isna()
        |
        (units["floor_area_sqm"] <= 0)
    )

    add_rejection_reason(
        units,
        invalid_floor_area,
        "invalid_floor_area"
    )


    units_clean, units_bad = (
        split_clean_and_quarantine(units)
    )


    valid_unit_ids = set(
        units_clean["unit_id"]
    )


    # ==================================
    # 3. LEASES
    # ==================================

    leases = datasets[
        "leases"
    ].copy()

    leases["rejection_reason"] = pd.NA


    duplicate_lease = leases.duplicated(
        subset=["lease_id"],
        keep=False
    )

    add_rejection_reason(
        leases,
        duplicate_lease,
        "duplicate_lease_id"
    )


    leases["start_date"] = pd.to_datetime(
        leases["start_date"],
        errors="coerce"
    )


    leases["end_date"] = pd.to_datetime(
        leases["end_date"],
        errors="coerce"
    )


    leases["monthly_rent_usd"] = pd.to_numeric(
        leases["monthly_rent_usd"],
        errors="coerce"
    )


    invalid_start_date = (
        leases["start_date"].isna()
    )

    add_rejection_reason(
        leases,
        invalid_start_date,
        "invalid_start_date"
    )


    invalid_date_order = (
        leases["end_date"].notna()
        &
        leases["start_date"].notna()
        &
        (
            leases["end_date"]
            <
            leases["start_date"]
        )
    )

    add_rejection_reason(
        leases,
        invalid_date_order,
        "end_date_before_start_date"
    )


    unknown_unit = (
        ~leases["unit_id"]
        .isin(valid_unit_ids)
    )

    add_rejection_reason(
        leases,
        unknown_unit,
        "unknown_unit_id"
    )


    invalid_rent = (
        leases["monthly_rent_usd"].isna()
        |
        (
            leases["monthly_rent_usd"]
            < 0
        )
    )

    add_rejection_reason(
        leases,
        invalid_rent,
        "invalid_monthly_rent"
    )


    leases_clean, leases_bad = (
        split_clean_and_quarantine(leases)
    )


    # ==================================
    # 4. WORK ORDERS
    # ==================================

    work_orders = datasets[
        "work_orders"
    ].copy()

    work_orders["rejection_reason"] = pd.NA


    duplicate_work_order = (
        work_orders.duplicated(
            subset=["work_order_id"],
            keep=False
        )
    )

    add_rejection_reason(
        work_orders,
        duplicate_work_order,
        "duplicate_work_order_id"
    )


    unknown_unit = (
        ~work_orders["unit_id"]
        .isin(valid_unit_ids)
    )

    add_rejection_reason(
        work_orders,
        unknown_unit,
        "unknown_unit_id"
    )


    # Adjust this name if required
    if "response_duration_hours" in work_orders.columns:

        work_orders[
            "response_duration_hours"
        ] = pd.to_numeric(
            work_orders[
                "response_duration_hours"
            ],
            errors="coerce"
        )


        missing_response = (
            work_orders[
                "response_duration_hours"
            ].isna()
        )

        add_rejection_reason(
            work_orders,
            missing_response,
            "missing_response_duration"
        )


        negative_response = (
            work_orders[
                "response_duration_hours"
            ] < 0
        )

        add_rejection_reason(
            work_orders,
            negative_response,
            "negative_response_duration"
        )


    # ==================================
    # 5. METER READINGS
    # ==================================

    meter_readings = datasets[
        "meter_readings"
    ].copy()

    meter_readings[
        "rejection_reason"
    ] = pd.NA


    meter_readings["energy_kwh"] = pd.to_numeric(
        meter_readings["energy_kwh"],
        errors="coerce"
    )


    invalid_kwh = (
        meter_readings["energy_kwh"].isna()
    )

    add_rejection_reason(
        meter_readings,
        invalid_kwh,
        "invalid_kwh"
    )


    negative_kwh = (
        meter_readings["energy_kwh"] < 0
    )

    add_rejection_reason(
        meter_readings,
        negative_kwh,
        "negative_kwh"
    )


    unknown_unit = (
        ~meter_readings["unit_id"]
        .isin(valid_unit_ids)
    )

    add_rejection_reason(
        meter_readings,
        unknown_unit,
        "unknown_unit_id"
    )


    if "reading_date" in meter_readings.columns:

        meter_readings[
            "reading_date"
        ] = pd.to_datetime(
            meter_readings[
                "reading_date"
            ],
            errors="coerce"
        )


        bad_date = (
            meter_readings[
                "reading_date"
            ].isna()
        )

        add_rejection_reason(
            meter_readings,
            bad_date,
            "invalid_reading_date"
        )


    work_orders_clean, work_orders_bad = (
        split_clean_and_quarantine(
            work_orders
        )
    )

    meter_clean, meter_bad = (
        split_clean_and_quarantine(
            meter_readings
        )
    )


    # ==================================
    # STORE RESULTS
    # ==================================

    clean_datasets = {

        "properties":
            properties_clean,

        "units":
            units_clean,

        "leases":
            leases_clean,

        "work_orders":
            work_orders_clean,

        "meter_readings":
            meter_clean
    }


    quarantine_datasets = {

        "properties":
            properties_bad,

        "units":
            units_bad,

        "leases":
            leases_bad,

        "work_orders":
            work_orders_bad,

        "meter_readings":
            meter_bad
    }


    # ==================================
    # QUALITY SUMMARY
    # ==================================

    for filename in datasets.keys():

        source_rows = len(
            datasets[filename]
        )

        clean_rows = len(
            clean_datasets[filename]
        )

        rejected_rows = len(
            quarantine_datasets[filename]
        )

        quality_summary[filename] = {

            "run_id":
                run_id,

            "source_rows":
                source_rows,

            "clean_rows":
                clean_rows,

            "rejected_rows":
                rejected_rows,

            "reconciliation_check":
                source_rows
                ==
                clean_rows
                + rejected_rows
        }


    return (
        clean_datasets,
        quarantine_datasets,
        quality_summary
    )

#### Run Validation

In [90]:
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

In [91]:
print(type(datasets))
print(datasets.keys())
print(run_id)
print(validate_and_quarantine)

<class 'dict'>
dict_keys(['properties', 'units', 'leases', 'work_orders', 'meter_readings'])
20261007_180849
<function validate_and_quarantine at 0x000001DF4B4020C0>


In [92]:
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print(run_id)

20261007_180905


In [94]:
clean_datasets, quarantine_datasets, quality_summary = (
    validate_and_quarantine(
        datasets=datasets,
        run_id=run_id
    )
)

In [95]:
def validate_and_quarantine(datasets: dict, run_id: str):

    clean_datasets = {}
    quarantine_datasets = {}
    quality_summary = {}

    # Validation logic goes here

    return (
        clean_datasets,
        quarantine_datasets,
        quality_summary
    )

#### Inspect the quality summary

In [96]:
quality_summary

{'properties': {'run_id': '20261007_180905',
  'source_rows': 12,
  'clean_rows': 12,
  'rejected_rows': 0,
  'reconciliation_check': True},
 'units': {'run_id': '20261007_180905',
  'source_rows': 241,
  'clean_rows': 239,
  'rejected_rows': 2,
  'reconciliation_check': True},
 'leases': {'run_id': '20261007_180905',
  'source_rows': 198,
  'clean_rows': 197,
  'rejected_rows': 1,
  'reconciliation_check': True},
 'work_orders': {'run_id': '20261007_180905',
  'source_rows': 1501,
  'clean_rows': 1489,
  'rejected_rows': 12,
  'reconciliation_check': True},
 'meter_readings': {'run_id': '20261007_180905',
  'source_rows': 1440,
  'clean_rows': 1432,
  'rejected_rows': 8,
  'reconciliation_check': True}}

In [97]:
quality_summary_df = pd.DataFrame(
    quality_summary
).T

quality_summary_df

,run_id,source_rows,clean_rows,rejected_rows,reconciliation_check
properties,20261007_180905,12,12,0,True
units,20261007_180905,241,239,2,True
leases,20261007_180905,198,197,1,True
work_orders,20261007_180905,1501,1489,12,True
meter_readings,20261007_180905,1440,1432,8,True


#### Inspect quarantined records

In [98]:
quarantine_datasets[
    "work_orders"
]

,work_order_id,unit_id,opened_date,closed_date,priority,response_hours,cost_usd,status,rejection_reason
8,WO000009,P008-U016,2026-02-27,2026-03-02,MEDIUM,69.7,153.97,RESOLVED,duplicate_work_order_id
34,WO000035,P999-U999,2026-04-08,NaN,LOW,NaN,82.75,OPEN,unknown_unit_id
70,WO000071,P001-U005,2026-01-04,2026-01-16,HIGH,35.8,393.80,RESOLVED,unknown_unit_id
93,WO000094,P001-U005,2026-06-01,2026-06-01,HIGH,52.0,965.55,RESOLVED,unknown_unit_id
278,WO000279,P001-U005,2026-06-23,2026-06-24,MEDIUM,78.4,514.39,RESOLVED,unknown_unit_id
469,WO000470,P001-U005,2026-01-17,2026-01-28,CRITICAL,23.9,422.12,RESOLVED,unknown_unit_id
515,WO000516,P001-U005,2026-06-02,2026-06-08,LOW,50.6,1143.80,RESOLVED,unknown_unit_id
602,WO000603,P001-U005,2026-03-04,2026-03-09,CRITICAL,56.0,201.89,RESOLVED,unknown_unit_id
1363,WO001364,P001-U005,2026-03-24,2026-03-30,HIGH,89.4,312.22,RESOLVED,unknown_unit_id
1398,WO001399,P001-U005,2026-03-06,2026-03-18,MEDIUM,50.9,917.91,RESOLVED,unknown_unit_id


In [99]:
quarantine_datasets[
    "meter_readings"
]

,reading_id,unit_id,reading_month,energy_kwh,rejection_reason
17,R000018,P001-U003,2026-06-01,-45.00,negative_kwh
24,R000025,P001-U005,2026-01-01,625.48,unknown_unit_id
25,R000026,P001-U005,2026-02-01,275.99,unknown_unit_id
26,R000027,P001-U005,2026-03-01,947.82,unknown_unit_id
27,R000028,P001-U005,2026-04-01,551.22,unknown_unit_id
28,R000029,P001-U005,2026-05-01,1089.05,unknown_unit_id
29,R000030,P001-U005,2026-06-01,258.96,unknown_unit_id
34,R000035,P999-U999,2026-05-01,1205.75,unknown_unit_id


In [100]:
quarantine_datasets[
    "units"
]

,unit_id,property_id,floor_area_sqm,active,rejection_reason
4,P001-U005,P001,150,1,duplicate_unit_id
240,P001-U005,P001,150,1,duplicate_unit_id


In [101]:
quarantine_datasets[
    "leases"
]

,lease_id,unit_id,start_date,end_date,monthly_rent_usd,rejection_reason
3,L00005,P001-U005,2025-03-16,NaT,975,unknown_unit_id


#### Test duplicate work order

In [102]:
work_order_quarantine = quarantine_datasets[
    "work_orders"
]

assert work_order_quarantine[
    "rejection_reason"
].str.contains(
    "duplicate_work_order_id"
).any()

print("PASS: Duplicate work order detected.")

PASS: Duplicate work order detected.


#### Test unknown unit 

In [103]:
all_quarantine = pd.concat(
    quarantine_datasets.values(),
    ignore_index=True
)

assert all_quarantine[
    "rejection_reason"
].astype(str).str.contains(
    "unknown_unit_id"
).any()

print("PASS: Unknown unit detected.")

PASS: Unknown unit detected.


##### Test negative energy_KWh 

In [105]:
meter_quarantine = quarantine_datasets[
    "meter_readings"
]

assert meter_quarantine[
    "rejection_reason"
].astype(str).str.contains(
    "negative_kwh"
).any()

print("PASS: Negative kWh detected.")

PASS: Negative kWh detected.


#### Test missing response

In [106]:
missing_response_mask = work_orders["response_hours"].isna()

work_orders.loc[
    missing_response_mask &
    work_orders["rejection_reason"].isna(),
    "rejection_reason"
] = "missing_response_duration"

print("PASS: Missing response detected.")

PASS: Missing response detected.


#### checking each datasets properties

In [107]:
for name, df in quarantine_datasets.items():
    print(f"\nDataset: {name}")

    if (
        not df.empty
        and "rejection_reason" in df.columns
    ):
        print(df["rejection_reason"].unique())


Dataset: properties

Dataset: units
['duplicate_unit_id']

Dataset: leases
['unknown_unit_id']

Dataset: work_orders
['duplicate_work_order_id' 'unknown_unit_id']

Dataset: meter_readings
['negative_kwh' 'unknown_unit_id']


#### Reconciliation Test

In [108]:
for filename, summary in quality_summary.items():

    assert (
        summary["source_rows"]
        ==
        summary["clean_rows"]
        +
        summary["rejected_rows"]
    )

    print(
        f"PASS: {filename} reconciles."
    )

PASS: properties reconciles.
PASS: units reconciles.
PASS: leases reconciles.
PASS: work_orders reconciles.
PASS: meter_readings reconciles.


#### Creating staging directory

In [109]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

STAGING_DIR = (
    PROJECT_ROOT
    / "data"
    / "staging"
)

QUARANTINE_DIR = (
    PROJECT_ROOT
    / "data"
    / "quarantine"
)

STAGING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QUARANTINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(STAGING_DIR.resolve())
print(QUARANTINE_DIR.resolve())

C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging
C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\quarantine


#### Save clean datasets

In [110]:
for filename, df in clean_datasets.items():

    output_path = (
        STAGING_DIR
        / filename
    )

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved clean dataset: "
        f"{output_path}"
    )

Saved clean dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\properties
Saved clean dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\units
Saved clean dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\leases
Saved clean dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\work_orders
Saved clean dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\meter_readings


#### Save quarantine datasets

In [111]:
for filename, df in quarantine_datasets.items():

    if df.empty:
        continue

    quarantine_path = (
        QUARANTINE_DIR
        /
        filename.replace(
            ".csv",
            "_quarantine.csv"
        )
    )

    df.to_csv(
        quarantine_path,
        index=False
    )

    print(
        f"Saved quarantine dataset: "
        f"{quarantine_path}"
    )

Saved quarantine dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\quarantine\units
Saved quarantine dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\quarantine\leases
Saved quarantine dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\quarantine\work_orders
Saved quarantine dataset: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\quarantine\meter_readings


#### Save quality summary

In [112]:
QUALITY_DIR = (
    PROJECT_ROOT
    / "data"
    / "curated"
)

QUALITY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

quality_summary_df = pd.DataFrame(
    quality_summary
).T

quality_summary_path = (
    QUALITY_DIR
    / "quality_summary.csv"
)

quality_summary_df.to_csv(
    quality_summary_path
)

print(
    "Quality summary saved to:",
    quality_summary_path.resolve()
)

Quality summary saved to: C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\curated\quality_summary.csv


#### Verify output

In [113]:
print("\nRAW DATA")
for file in (
    PROJECT_ROOT
    / "data"
    / "raw"
).glob("*"):
    print(file.name)


print("\nSTAGING DATA")
for file in STAGING_DIR.glob("*"):
    print(file.name)


print("\nQUARANTINE DATA")
for file in QUARANTINE_DIR.glob("*"):
    print(file.name)


RAW DATA
leases.csv
meter_readings.csv
properties.csv
units.csv
work_orders.csv

STAGING DATA
leases
meter_readings
properties
units
work_orders

QUARANTINE DATA
leases
meter_readings
units
work_orders


#### checking the actual coulmn names on each datadets

In [114]:
for filename, df in datasets.items():

    print("\n", filename)

    print(
        df.columns.tolist()
    )


 properties
['property_id', 'property_name', 'city']

 units
['unit_id', 'property_id', 'floor_area_sqm', 'active']

 leases
['lease_id', 'unit_id', 'start_date', 'end_date', 'monthly_rent_usd']

 work_orders
['work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status']

 meter_readings
['reading_id', 'unit_id', 'reading_month', 'energy_kwh']


#### Defining my project paths

In [115]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

STAGING_DIR = PROJECT_ROOT / "data" / "staging"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Staging directory:", STAGING_DIR)
print("Curated directory:", CURATED_DIR)

Project root: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline
Staging directory: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging
Curated directory: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\curated


#### Confirming staging files exist

In [116]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

In [ ]:
from pathlib import Path

STAGING_DIR = Path("data/staging")

print("Current working directory:")
print(Path.cwd())

print("\nStaging directory:")
print(STAGING_DIR.resolve())

print("\nFiles in staging:")
for file in STAGING_DIR.glob("*"):
    print(file.name)

Current working directory:
c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\notebook

Staging directory:
C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\notebook\data\staging

Files in staging:


In [117]:
STAGING_DIR.mkdir(parents=True, exist_ok=True)

clean_datasets["properties"].to_csv(
    STAGING_DIR / "properties.csv",
    index=False
)

clean_datasets["units"].to_csv(
    STAGING_DIR / "units.csv",
    index=False
)

clean_datasets["leases"].to_csv(
    STAGING_DIR / "leases.csv",
    index=False
)

clean_datasets["work_orders"].to_csv(
    STAGING_DIR / "work_orders.csv",
    index=False
)

clean_datasets["meter_readings"].to_csv(
    STAGING_DIR / "meter_readings.csv",
    index=False
)

print("Clean datasets saved to staging.")

Clean datasets saved to staging.


#### check the staging file actually exist

In [118]:
for file in STAGING_DIR.glob("*.csv"):
    print(file)

c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\leases.csv
c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\meter_readings.csv
c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\properties.csv
c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\units.csv
c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\staging\work_orders.csv


#### Read validated staging data with spark

In [120]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("PropTech360")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.2.0


In [121]:
properties = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str((STAGING_DIR / "properties.csv").resolve()))
)

units = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str((STAGING_DIR / "units.csv").resolve()))
)

leases = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str((STAGING_DIR / "leases.csv").resolve()))
)

work_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str((STAGING_DIR / "work_orders.csv").resolve()))
)

meter_readings = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str((STAGING_DIR / "meter_readings.csv").resolve()))
)

#### Checking Schemas

In [122]:
print("PROPERTIES")
properties.printSchema()

print("UNITS")
units.printSchema()

print("LEASES")
leases.printSchema()

print("WORK ORDERS")
work_orders.printSchema()

print("METER READINGS")
meter_readings.printSchema()

PROPERTIES
root
 |-- property_id: string (nullable = true)
 |-- property_name: string (nullable = true)
 |-- city: string (nullable = true)

UNITS
root
 |-- unit_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- floor_area_sqm: integer (nullable = true)
 |-- active: integer (nullable = true)

LEASES
root
 |-- lease_id: string (nullable = true)
 |-- unit_id: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- monthly_rent_usd: integer (nullable = true)

WORK ORDERS
root
 |-- work_order_id: string (nullable = true)
 |-- unit_id: string (nullable = true)
 |-- opened_date: date (nullable = true)
 |-- closed_date: date (nullable = true)
 |-- priority: string (nullable = true)
 |-- response_hours: double (nullable = true)
 |-- cost_usd: double (nullable = true)
 |-- status: string (nullable = true)

METER READINGS
root
 |-- reading_id: string (nullable = true)
 |-- unit_id: string (nullable = true)
 |-- readin

#### check the actual coulmns names

In [123]:
for name, df in {
    "properties": properties,
    "units": units,
    "leases": leases,
    "work_orders": work_orders,
    "meter_readings": meter_readings
}.items():

    print(name)
    print(df.columns)
    print()

properties
['property_id', 'property_name', 'city']

units
['unit_id', 'property_id', 'floor_area_sqm', 'active']

leases
['lease_id', 'unit_id', 'start_date', 'end_date', 'monthly_rent_usd']

work_orders
['work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status']

meter_readings
['reading_id', 'unit_id', 'reading_month', 'energy_kwh']



#### Standardise all the datatypes

#### Properties

In [124]:
properties = (
    properties
    .withColumn(
        "property_id",
        F.col("property_id").cast("string")
    )
)

#### Units

In [125]:
units = (
    units
    .withColumn(
        "unit_id",
        F.col("unit_id").cast("string")
    )
    .withColumn(
        "property_id",
        F.col("property_id").cast("string")
    )
    .withColumn(
        "floor_area_sqm",
        F.col("floor_area_sqm").cast("double")
    )
)

#### Leases

In [126]:
leases = (
    leases
    .withColumn(
        "lease_id",
        F.col("lease_id").cast("string")
    )
    .withColumn(
        "unit_id",
        F.col("unit_id").cast("string")
    )
    .withColumn(
        "start_date",
        F.to_date("start_date")
    )
    .withColumn(
        "end_date",
        F.to_date("end_date")
    )
    .withColumn(
        "monthly_rent_usd",
        F.col("monthly_rent_usd").cast("double")
    )
)

##### Work_orders

In [129]:
work_orders = (
    work_orders
    .withColumn(
        "work_order_id",
        F.col("work_order_id").cast("string")
    )
    .withColumn(
        "unit_id",
        F.col("unit_id").cast("string")
    )
    .withColumn(
        "opened_date",
        F.to_date("opened_date")
    )
    .withColumn(
        "response_hours",
        F.col(
            "response_hours"
        ).cast("double")
    )
)

##### Meter_readings

In [131]:
meter_readings = (
    meter_readings
    .withColumn(
        "reading_id",
        F.col("reading_id").cast("string")
    )
    .withColumn(
        "reading_date",
        F.to_date("reading_month")
    )
    .withColumn(
        "kwh",
        F.col("energy_kwh").cast("double")
    )
)

##### Creating six-month Calender with month-end dates

##### A reproducible approach is to derive the reporting period from the data rather than today's date

##### Earliest lease month starts

In [132]:
report_start = (
    leases
    .select(
        F.trunc(
            F.min("start_date"),
            "month"
        ).alias("report_start")
    )
    .first()["report_start"]
)

print("Reporting start:", report_start)

Reporting start: 2025-01-01


In [134]:
REPORT_START = "2025-01-01"
REPORT_END = "2025-06-30"

In [135]:
calendar = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(
                    F.lit(REPORT_START)
                ),
                F.add_months(
                    F.to_date(
                        F.lit(REPORT_START)
                    ),
                    5
                ),
                F.expr("interval 1 month")
            )
        ).alias("month_start")
    )
)

##### calculate month-end

In [136]:
calendar = (
    calendar
    .withColumn(
        "month_end",
        F.last_day("month_start")
    )
)

In [137]:
calendar.show()

+-----------+----------+
|month_start| month_end|
+-----------+----------+
| 2025-01-01|2025-01-31|
| 2025-02-01|2025-02-28|
| 2025-03-01|2025-03-31|
| 2025-04-01|2025-04-30|
| 2025-05-01|2025-05-31|
| 2025-06-01|2025-06-30|
+-----------+----------+



#####  confirming the six-months calender has been created

In [138]:
assert calendar.count() == 6

print("PASS: Six-month calendar created.")

PASS: Six-month calendar created.


##### Creating all the property-month combination

In [139]:
property_count = (
    properties
    .select("property_id")
    .distinct()
    .count()
)

print("Number of properties:", property_count)

Number of properties: 12


##### create the property-month framework

In [140]:
property_months = (
    properties
    .select(
        "property_id",
        "property_name",
        "city"
    )
    .distinct()
    .crossJoin(calendar)
)

##### checking

In [141]:
print(
    "Property-month rows:",
    property_months.count()
)

Property-month rows: 72


##### creating unit-month base

In [142]:
unit_months = (
    units
    .select(
        "unit_id",
        "property_id"
    )
    .crossJoin(calendar)
)

##### Join Leases to Unit-months

In [143]:
unit_month_leases = (
    unit_months.alias("um")
    .join(
        leases.alias("l"),
        (
            F.col("um.unit_id")
            ==
            F.col("l.unit_id")
        )
        &
        (
            F.col("l.start_date")
            <=
            F.col("um.month_end")
        )
        &
        (
            F.col("l.end_date").isNull()
            |
            (
                F.col("l.end_date")
                >=
                F.col("um.month_end")
            )
        ),
        "left"
    )
)

##### Detecting overlapping lease

In [144]:
lease_overlap_check = (
    unit_month_leases
    .groupBy(
        F.col("um.unit_id").alias("unit_id"),
        F.col("um.month_end").alias("month_end")
    )
    .agg(
        F.count(
            F.col("l.lease_id")
        ).alias("active_lease_count")
    )
)

##### find overlaps

In [145]:
lease_overlaps = (
    lease_overlap_check
    .filter(
        F.col("active_lease_count") > 1
    )
)

#### checking overlaps

In [146]:
print(
    "Lease overlap rows:",
    lease_overlaps.count()
)

Lease overlap rows: 0


##### view the table

In [147]:
lease_overlaps.show(
    truncate=False
)

+-------+---------+------------------+
|unit_id|month_end|active_lease_count|
+-------+---------+------------------+
+-------+---------+------------------+



##### create a window

##### keep the lease with the start date for that unit/month

In [148]:
lease_window = (
    Window
    .partitionBy(
        F.col("um.unit_id"),
        F.col("um.month_end")
    )
    .orderBy(
        F.col("l.start_date").desc(),
        F.col("l.lease_id").desc()
    )
)

##### Apply row number

In [149]:
unit_month_ranked = (
    unit_month_leases
    .withColumn(
        "lease_rank",
        F.row_number().over(
            lease_window
        )
    )
)

##### keep one lease per unit-month

In [150]:
unit_month_snapshot = (
    unit_month_ranked
    .filter(
        F.col("lease_rank") == 1
    )
    .select(
        F.col("um.unit_id").alias("unit_id"),
        F.col("um.property_id").alias("property_id"),
        F.col("um.month_start").alias("month_start"),
        F.col("um.month_end").alias("month_end"),

        F.col("l.lease_id").alias("lease_id"),

        F.col(
            "l.monthly_rent_usd"
        ).alias("monthly_rent_usd")
    )
)

##### Create occupancy indicator

In [151]:
unit_month_snapshot = (
    unit_month_snapshot
    .withColumn(
        "occupied_flag",
        F.when(
            F.col("lease_id").isNotNull(),
            F.lit(1)
        ).otherwise(
            F.lit(0)
        )
    )
)

#### checking the snapshots

In [152]:
unit_month_snapshot.show()

+---------+-----------+-----------+----------+--------+----------------+-------------+
|  unit_id|property_id|month_start| month_end|lease_id|monthly_rent_usd|occupied_flag|
+---------+-----------+-----------+----------+--------+----------------+-------------+
|P001-U001|       P001| 2025-01-01|2025-01-31|    NULL|            NULL|            0|
|P001-U001|       P001| 2025-02-01|2025-02-28|    NULL|            NULL|            0|
|P001-U001|       P001| 2025-03-01|2025-03-31|  L00001|           975.0|            1|
|P001-U001|       P001| 2025-04-01|2025-04-30|  L00001|           975.0|            1|
|P001-U001|       P001| 2025-05-01|2025-05-31|  L00001|           975.0|            1|
|P001-U001|       P001| 2025-06-01|2025-06-30|  L00001|           975.0|            1|
|P001-U002|       P001| 2025-01-01|2025-01-31|    NULL|            NULL|            0|
|P001-U002|       P001| 2025-02-01|2025-02-28|    NULL|            NULL|            0|
|P001-U002|       P001| 2025-03-01|2025-03-

#### Calculate property-month occupancy INDEPENDENTLY

##### Do not join the work orders and meter readings directly to individual lease records at the beginning, as this can create duplicate rows and inflate the data.

#### Instead, calculate the results for each business area independently before combining them.


##### FOR occupancy/rent

In [153]:
occupancy_rent = (
    unit_month_snapshot
    .groupBy(
        "property_id",
        "month_end"
    )
    .agg(
        F.countDistinct(
            "unit_id"
        ).alias("total_units"),

        F.sum(
            "occupied_flag"
        ).alias("occupied_units"),

        F.sum(
            F.when(
                F.col("occupied_flag") == 1,
                F.col("monthly_rent_usd")
            ).otherwise(
                F.lit(0.0)
            )
        ).alias("monthly_rent_usd")
    )
)

#### Calculate occupancy rate

In [154]:
occupancy_rent = (
    occupancy_rent
    .withColumn(
        "occupancy_rate",
        F.when(
            F.col("total_units") > 0,
            F.col("occupied_units")
            /
            F.col("total_units")
        ).otherwise(
            F.lit(0.0)
        )
    )
)

#### You can also express the result as a percentage

In [155]:
occupancy_rent = (
    occupancy_rent
    .withColumn(
        "occupancy_pct",
        F.round(
            F.col("occupancy_rate") * 100,
            2
        )
    )
)

In [156]:
occupancy_rent.show()

+-----------+----------+-----------+--------------+----------------+-------------------+-------------+
|property_id| month_end|total_units|occupied_units|monthly_rent_usd|     occupancy_rate|occupancy_pct|
+-----------+----------+-----------+--------------+----------------+-------------------+-------------+
|       P002|2025-02-28|         20|             4|          5875.0|                0.2|         20.0|
|       P006|2025-04-30|         20|            11|         18600.0|               0.55|         55.0|
|       P008|2025-02-28|         20|             6|          7950.0|                0.3|         30.0|
|       P006|2025-02-28|         20|             3|          5900.0|               0.15|         15.0|
|       P011|2025-02-28|         20|             4|          5325.0|                0.2|         20.0|
|       P002|2025-03-31|         20|             5|          6675.0|               0.25|         25.0|
|       P010|2025-02-28|         20|             1|           975.0|     

#### Prepare worder_orders independently

##### First, link each work order to the correct property using the unit information:

In [157]:
work_orders_enriched = (
    work_orders.alias("w")
    .join(
        units.select(
            "unit_id",
            "property_id"
        ).alias("u"),
        F.col("w.unit_id")
        ==
        F.col("u.unit_id"),
        "inner"
    )
)

#### Creating month

In [158]:
work_orders_enriched = (
    work_orders_enriched
    .withColumn(
        "month_end",
        F.last_day(
            F.col("opened_date")
        )
    )
)

#### Defien Service Level Agreement (SLA)

##### You need to define a clear SLA time limit

In [159]:
SLA_HOURS = 24

##### After that:

In [160]:
work_orders_enriched = (
    work_orders_enriched
    .withColumn(
        "sla_met",
        F.when(
            F.col(
                "response_hours"
            )
            <= SLA_HOURS,
            F.lit(1)
        ).otherwise(
            F.lit(0)
        )
    )
)

#### Aggregate SLA independently

In [161]:
sla_metrics = (
    work_orders_enriched
    .groupBy(
        "property_id",
        "month_end"
    )
    .agg(
        F.count(
            "*"
        ).alias("work_order_count"),

        F.avg(
            "response_hours"
        ).alias(
            "avg_response_hours"
        ),

        F.sum(
            "sla_met"
        ).alias(
            "sla_met_count"
        )
    )
)

##### Then calculate SLA percentage

In [162]:
sla_metrics = (
    sla_metrics
    .withColumn(
        "sla_pct",
        F.when(
            F.col("work_order_count") > 0,

            F.round(
                F.col("sla_met_count")
                /
                F.col("work_order_count")
                * 100,
                2
            )
        ).otherwise(
            F.lit(None)
        )
    )
)

#### Check

In [163]:
sla_metrics.show()

+-----------+----------+----------------+------------------+-------------+-------+
|property_id| month_end|work_order_count|avg_response_hours|sla_met_count|sla_pct|
+-----------+----------+----------------+------------------+-------------+-------+
|       P004|2026-03-31|              13|             43.17|            2|  15.38|
|       P005|2026-05-31|              16| 53.00769230769231|            3|  18.75|
|       P006|2026-05-31|              20|55.228571428571435|            1|    5.0|
|       P010|2026-02-28|              17| 58.54615384615384|            2|  11.76|
|       P001|2026-04-30|              15| 56.31818181818183|            1|   6.67|
|       P007|2026-02-28|              31|39.634482758620685|           13|  41.94|
|       P009|2026-06-30|              15|48.718181818181826|            2|  13.33|
|       P002|2026-04-30|              27|51.777272727272724|            4|  14.81|
|       P010|2026-05-31|              24| 51.19500000000001|            5|  20.83|
|   

##### Prepare meter readings independently

##### Link the meter readings to the corresponding units:

In [164]:
energy_enriched = (
    meter_readings.alias("m")
    .join(
        units.select(
            "unit_id",
            "property_id"
        ).alias("u"),
        F.col("m.unit_id")
        ==
        F.col("u.unit_id"),
        "inner"
    )
)

##### Add Month-end

In [165]:
energy_enriched = (
    energy_enriched
    .withColumn(
        "month_end",
        F.last_day(
            F.col("reading_date")
        )
    )
)

#### Calculate property-month energy independently

In [166]:
energy_metrics = (
    energy_enriched
    .groupBy(
        "property_id",
        "month_end"
    )
    .agg(
        F.sum(
            "kwh"
        ).alias(
            "total_energy_kwh"
        ),

        F.avg(
            "kwh"
        ).alias(
            "avg_reading_kwh"
        ),

        F.count(
            "*"
        ).alias(
            "meter_reading_count"
        )
    )
)

##### check

In [167]:
energy_metrics.show()

+-----------+----------+------------------+-----------------+-------------------+
|property_id| month_end|  total_energy_kwh|  avg_reading_kwh|meter_reading_count|
+-----------+----------+------------------+-----------------+-------------------+
|       P004|2026-03-31|12853.620000000003|642.6810000000002|                 20|
|       P005|2026-05-31|16005.140000000001|800.2570000000001|                 20|
|       P006|2026-05-31|           13575.5|          678.775|                 20|
|       P010|2026-02-28|          14572.61|         728.6305|                 20|
|       P001|2026-04-30|           14376.2| 756.642105263158|                 19|
|       P007|2026-02-28|14900.219999999998|745.0109999999999|                 20|
|       P009|2026-06-30|16448.699999999997|822.4349999999998|                 20|
|       P002|2026-04-30|12719.989999999998|635.9994999999999|                 20|
|       P010|2026-05-31|14956.550000000003|747.8275000000001|                 20|
|       P009|202

##### Confirm thre three independent aggregates

In [168]:
occupancy_rent

DataFrame[property_id: string, month_end: date, total_units: bigint, occupied_units: bigint, monthly_rent_usd: double, occupancy_rate: double, occupancy_pct: double]

In [169]:
energy_metrics

DataFrame[property_id: string, month_end: date, total_energy_kwh: double, avg_reading_kwh: double, meter_reading_count: bigint]

#### Start the final output using the property-month base table

In [170]:
property_months

DataFrame[property_id: string, property_name: string, city: string, month_start: date, month_end: date]

#### Start with:

In [171]:
property_month_kpis = (
    property_months
    .select(
        "property_id",
        "property_name",
        "city",
        "month_start",
        "month_end"
    )
)

##### Join Occupancy

In [172]:
property_month_kpis = (
    property_month_kpis
    .join(
        occupancy_rent,
        on=[
            "property_id",
            "month_end"
        ],
        how="left"
    )
)

#### Join SLA metrics

In [173]:
property_month_kpis = (
    property_month_kpis
    .join(
        sla_metrics,
        on=[
            "property_id",
            "month_end"
        ],
        how="left"
    )
)

#### Join energy metrics

In [174]:
property_month_kpis = (
    property_month_kpis
    .join(
        energy_metrics,
        on=[
            "property_id",
            "month_end"
        ],
        how="left"
    )
)

#### Fill appropriate missing values

##### For count/total fields, use zero where logically appropriate

In [175]:
property_month_kpis = (
    property_month_kpis
    .fillna(
        {
            "total_units": 0,
            "occupied_units": 0,
            "monthly_rent_usd": 0.0,
            "occupancy_rate": 0.0,
            "occupancy_pct": 0.0,
            "work_order_count": 0,
            "sla_met_count": 0,
            "total_energy_kwh": 0.0,
            "meter_reading_count": 0
        }
    )
)

#### Round business metrics

In [176]:
property_month_kpis = (
    property_month_kpis
    .withColumn(
        "monthly_rent_usd",
        F.round(
            "monthly_rent_usd",
            2
        )
    )
    .withColumn(
        "avg_response_hours",
        F.round(
            "avg_response_hours",
            2
        )
    )
    .withColumn(
        "total_energy_kwh",
        F.round(
            "total_energy_kwh",
            2
        )
    )
    .withColumn(
        "avg_reading_kwh",
        F.round(
            "avg_reading_kwh",
            2
        )
    )
)

#### Organise the results

In [177]:
property_month_kpis = (
    property_month_kpis
    .orderBy(
        "property_id",
        "month_end"
    )
)

In [178]:
property_month_kpis.show(
    100,
    truncate=False
)

+-----------+----------+-------------------+-------------+-----------+-----------+--------------+----------------+-------------------+-------------+----------------+------------------+-------------+-------+----------------+---------------+-------------------+
|property_id|month_end |property_name      |city         |month_start|total_units|occupied_units|monthly_rent_usd|occupancy_rate     |occupancy_pct|work_order_count|avg_response_hours|sla_met_count|sla_pct|total_energy_kwh|avg_reading_kwh|meter_reading_count|
+-----------+----------+-------------------+-------------+-----------+-----------+--------------+----------------+-------------------+-------------+----------------+------------------+-------------+-------+----------------+---------------+-------------------+
|P001       |2025-01-31|MetroSpace Tower 01|Lagos        |2025-01-01 |19         |1             |800.0           |0.05263157894736842|5.26         |0               |NULL              |0            |NULL   |0.0           

#### Assert exactly 72 rows

##### First count rows:

In [179]:
row_count = property_month_kpis.count()

print("Total rows:", row_count)

Total rows: 72


In [180]:
assert row_count == 72, (
    f"Expected 72 rows, "
    f"but found {row_count}"
)

print("PASS: Exactly 72 rows.")

PASS: Exactly 72 rows.


#### Assert Uniqueness

In [181]:
unique_count = (
    property_month_kpis
    .select(
        "property_id",
        "month_end"
    )
    .distinct()
    .count()
)

##### assertion

In [182]:
assert unique_count == 72, (
    f"Expected 72 unique "
    f"property-month combinations, "
    f"but found {unique_count}"
)

print(
    "PASS: 72 unique property-month rows."
)

PASS: 72 unique property-month rows.


#### Check for duplicates 

In [183]:
duplicate_property_months = (
    property_month_kpis
    .groupBy(
        "property_id",
        "month_end"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

In [184]:
assert duplicate_property_months.count() == 0

print(
    "PASS: No duplicate "
    "property-month records."
)

PASS: No duplicate property-month records.


#### Validate expected six months

In [185]:
month_count = (
    property_month_kpis
    .select("month_end")
    .distinct()
    .count()
)

assert month_count == 6

print("PASS: Exactly six months.")

PASS: Exactly six months.


#### Validate expected properties

In [186]:
property_count = (
    property_month_kpis
    .select("property_id")
    .distinct()
    .count()
)

print(
    "Distinct properties:",
    property_count
)

Distinct properties: 12


In [187]:
assert property_count * month_count == 72

print(
    "PASS:",
    property_count,
    "properties ×",
    month_count,
    "months = 72"
)

PASS: 12 properties × 6 months = 72


#### Add a partition column

##### Write a Parquet partitioned by month

##### Create

In [188]:
property_month_kpis = (
    property_month_kpis
    .withColumn(
        "report_month",
        F.date_format(
            "month_end",
            "yyyy-MM"
        )
    )
)

#### Define Parquet output path

In [189]:
PARQUET_OUTPUT = (
    CURATED_DIR
    / "property_month_kpis"
)

print(PARQUET_OUTPUT)

c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\curated\property_month_kpis


In [211]:
from pathlib import Path

PARQUET_OUTPUT = Path(
    "data/processed/property_month_kpis"
).resolve()

PARQUET_OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("Parquet output path:")
print(PARQUET_OUTPUT)

Parquet output path:
C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\notebook\data\processed\property_month_kpis


In [212]:
property_month_kpis

DataFrame[property_id: string, month_end: date, property_name: string, city: string, month_start: date, total_units: bigint, occupied_units: bigint, monthly_rent_usd: double, occupancy_rate: double, occupancy_pct: double, work_order_count: bigint, avg_response_hours: double, sla_met_count: bigint, sla_pct: double, total_energy_kwh: double, avg_reading_kwh: double, meter_reading_count: bigint, report_month: string]

In [213]:
print(property_month_kpis.columns)

['property_id', 'month_end', 'property_name', 'city', 'month_start', 'total_units', 'occupied_units', 'monthly_rent_usd', 'occupancy_rate', 'occupancy_pct', 'work_order_count', 'avg_response_hours', 'sla_met_count', 'sla_pct', 'total_energy_kwh', 'avg_reading_kwh', 'meter_reading_count', 'report_month']


In [214]:
property_month_kpis.printSchema()

root
 |-- property_id: string (nullable = true)
 |-- month_end: date (nullable = false)
 |-- property_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- month_start: date (nullable = false)
 |-- total_units: long (nullable = false)
 |-- occupied_units: long (nullable = false)
 |-- monthly_rent_usd: double (nullable = true)
 |-- occupancy_rate: double (nullable = false)
 |-- occupancy_pct: double (nullable = false)
 |-- work_order_count: long (nullable = false)
 |-- avg_response_hours: double (nullable = true)
 |-- sla_met_count: long (nullable = false)
 |-- sla_pct: double (nullable = true)
 |-- total_energy_kwh: double (nullable = true)
 |-- avg_reading_kwh: double (nullable = true)
 |-- meter_reading_count: long (nullable = false)
 |-- report_month: string (nullable = false)



In [215]:
if "report_month" not in property_month_kpis.columns:
    raise ValueError(
        "The column 'report_month' is missing from property_month_kpis."
    )

In [217]:
type(property_month_kpis)

pyspark.sql.classic.dataframe.DataFrame

In [219]:
print(property_month_kpis.columns)

['property_id', 'month_end', 'property_name', 'city', 'month_start', 'total_units', 'occupied_units', 'monthly_rent_usd', 'occupancy_rate', 'occupancy_pct', 'work_order_count', 'avg_response_hours', 'sla_met_count', 'sla_pct', 'total_energy_kwh', 'avg_reading_kwh', 'meter_reading_count', 'report_month']


In [220]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [228]:
property_month_kpis.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(report_month, 'date_format('month_end, yyyy-MM), None)]
+- Sort [property_id#110 ASC NULLS FIRST, month_end#151 ASC NULLS FIRST], true
   +- Project [property_id#110, month_end#151, property_name#18, city#19, month_start#150, total_units#513L, occupied_units#514L, monthly_rent_usd#522, occupancy_rate#516, occupancy_pct#517, work_order_count#518L, avg_response_hours#523, sla_met_count#519L, sla_pct#398, total_energy_kwh#524, round(avg_reading_kwh#442, 2) AS avg_reading_kwh#525, meter_reading_count#521L]
      +- Project [property_id#110, month_end#151, property_name#18, city#19, month_start#150, total_units#513L, occupied_units#514L, monthly_rent_usd#522, occupancy_rate#516, occupancy_pct#517, work_order_count#518L, avg_response_hours#523, sla_met_count#519L, sla_pct#398, round(total_energy_kwh#520, 2) AS total_energy_kwh#524, avg_reading_kwh#442, meter_reading_count#521L]
         +- Project [property_id#110, month_end#151, 

In [227]:
property_month_kpis.select("report_month").distinct().show()

Py4JJavaError: An error occurred while calling o875.showString.
: java.lang.NullPointerException: Cannot invoke "org.apache.spark.scheduler.DAGScheduler.cleanupQueryJobs(long)" because the return value of "org.apache.spark.SparkContext.dagScheduler()" is null
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId0$8(SQLExecution.scala:289)
	at org.apache.spark.sql.execution.SQLExecution$.withSessionTagsApplied(SQLExecution.scala:352)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId0$7(SQLExecution.scala:189)
	at org.apache.spark.JobArtifactSet$.withActiveJobArtifactState(JobArtifactSet.scala:94)
	at org.apache.spark.sql.artifact.ArtifactManager.$anonfun$withResources$1(ArtifactManager.scala:112)
	at org.apache.spark.sql.artifact.ArtifactManager.withClassLoaderIfNeeded(ArtifactManager.scala:106)
	at org.apache.spark.sql.artifact.ArtifactManager.withResources(ArtifactManager.scala:111)
	at org.apache.spark.sql.execution.SQLExecuti

In [226]:
property_month_kpis.select("report_month").limit(5).show()

Py4JJavaError: An error occurred while calling o868.showString.
: java.lang.NullPointerException: Cannot invoke "org.apache.spark.scheduler.DAGScheduler.cleanupQueryJobs(long)" because the return value of "org.apache.spark.SparkContext.dagScheduler()" is null
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId0$8(SQLExecution.scala:289)
	at org.apache.spark.sql.execution.SQLExecution$.withSessionTagsApplied(SQLExecution.scala:352)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId0$7(SQLExecution.scala:189)
	at org.apache.spark.JobArtifactSet$.withActiveJobArtifactState(JobArtifactSet.scala:94)
	at org.apache.spark.sql.artifact.ArtifactManager.$anonfun$withResources$1(ArtifactManager.scala:112)
	at org.apache.spark.sql.artifact.ArtifactManager.withClassLoaderIfNeeded(ArtifactManager.scala:106)
	at org.apache.spark.sql.artifact.ArtifactManager.withResources(ArtifactManager.scala:111)
	at org.apache.spark.sql.execution.SQLExecuti

In [216]:
property_month_kpis.select(
    "report_month"
).distinct().show()

Py4JJavaError: An error occurred while calling o847.showString.
: java.lang.NullPointerException: Cannot invoke "org.apache.spark.scheduler.DAGScheduler.cleanupQueryJobs(long)" because the return value of "org.apache.spark.SparkContext.dagScheduler()" is null
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId0$8(SQLExecution.scala:289)
	at org.apache.spark.sql.execution.SQLExecution$.withSessionTagsApplied(SQLExecution.scala:352)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId0$7(SQLExecution.scala:189)
	at org.apache.spark.JobArtifactSet$.withActiveJobArtifactState(JobArtifactSet.scala:94)
	at org.apache.spark.sql.artifact.ArtifactManager.$anonfun$withResources$1(ArtifactManager.scala:112)
	at org.apache.spark.sql.artifact.ArtifactManager.withClassLoaderIfNeeded(ArtifactManager.scala:106)
	at org.apache.spark.sql.artifact.ArtifactManager.withResources(ArtifactManager.scala:111)
	at org.apache.spark.sql.execution.SQLExecuti

In [205]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [206]:
property_month_kpis.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(report_month, 'date_format('month_end, yyyy-MM), None)]
+- Sort [property_id#110 ASC NULLS FIRST, month_end#151 ASC NULLS FIRST], true
   +- Project [property_id#110, month_end#151, property_name#18, city#19, month_start#150, total_units#513L, occupied_units#514L, monthly_rent_usd#522, occupancy_rate#516, occupancy_pct#517, work_order_count#518L, avg_response_hours#523, sla_met_count#519L, sla_pct#398, total_energy_kwh#524, round(avg_reading_kwh#442, 2) AS avg_reading_kwh#525, meter_reading_count#521L]
      +- Project [property_id#110, month_end#151, property_name#18, city#19, month_start#150, total_units#513L, occupied_units#514L, monthly_rent_usd#522, occupancy_rate#516, occupancy_pct#517, work_order_count#518L, avg_response_hours#523, sla_met_count#519L, sla_pct#398, round(total_energy_kwh#520, 2) AS total_energy_kwh#524, avg_reading_kwh#442, meter_reading_count#521L]
         +- Project [property_id#110, month_end#151, 

In [198]:
import os

print(os.environ["HADOOP_HOME"])
print(os.path.exists(r"C:\hadoop\bin\winutils.exe"))

C:\hadoop
False
